<a href="https://colab.research.google.com/github/najwatul1224-sys/tugasdeeplearning/blob/main/project_deep_learing_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Deteksi Toxic Comment Multi-Kelas Menggunakan LSTM, GRU, dan indoBERT dengan Analisis Perbandingan Performa Model**

In [ ]:
!pip install -U transformers tensorflow tf-keras

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
data = pd.read_csv('dataset_tahunsemua.csv')
data.head()

**PREPROCESSING DATA**

In [ ]:
data.info()

In [ ]:
df = pd.DataFrame(data['full_text'].iloc[0:613])
df.head()

**CLEANING**

In [ ]:
import re
import string
import nltk

def remove_URL(tweet):
  url = re.compile(r'https?://\S+|www\.\S+')
  return url.sub(r'',tweet)

def remove_html(tweet):
  html = re.compile(r'<.*?>')
  return html.sub(r'',tweet)

def remove_emoji(tweet):
  emoji_pattern = re.compile("["
      u"\U0001F600-\U0001F64F"
      u"\U0001F300-\U0001F5FF"
      u"\U0001F680-\U0001F6FF"
      u"\U0001F1E0-\U0001F1FF"
                        "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r'',tweet)

def remove_numbers(tweet):
  tweet = re.sub(r'\d+', '', tweet)
  return tweet

def remove_symbols(tweet):
  tweet = re.sub(r'[^a-zA-Z0-9\s]', '',tweet)
  return tweet

df['cleansing'] = df['full_text'].apply(lambda x: remove_URL(x))
df['cleansing'] = df['full_text'].apply(lambda x: remove_html(x))
df['cleansing'] = df['full_text'].apply(lambda x: remove_emoji(x))
df['cleansing'] = df['full_text'].apply(lambda x: remove_symbols(x))
df['cleansing'] = df['full_text'].apply(lambda x: remove_numbers(x))

df.head()

**CASE FOLDING**

In [ ]:
def case_folding(text):
  if isinstance(text, str):
    lowercase_text = text.lower()
    return lowercase_text
  else:
    return text

df['case_folding'] = df['cleansing'].apply(case_folding)

df.head()

**TOKENIZATION**

In [ ]:
def tokenize(text):
  tokens = text.split()
  return tokens

df['tokenize'] = df['case_folding'].apply(tokenize)

df.head()

**FILTERING/STOPWORD REMOVAL**

---



In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = stopwords.words('indonesian')

def remove_stopwords(text):
  return [word for word in text if word not in stop_words]

df['Filtering/stopword removal'] = df['tokenize'].apply(lambda x: remove_stopwords(x))

df.head()

In [ ]:
!pip install Sastrawi

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.stem import PorterStemmer
from nltk.stem.snowball import SnowballStemmer

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_text(text):
  return [stemmer.stem(word) for word in text]

df['stemming_data'] = df['Filtering/stopword removal'].apply(lambda x: ' '.join(stem_text(x)))

df.head()

In [ ]:
df.to_csv('hasil_preprocessing.csv', encoding='utf8', index=False)

**PELABELAN**

In [ ]:
import pandas as pd
import numpy as np

def load_data():
  data = pd.read_csv('hasil_preprocessing.csv')
  return data

data = load_data()
data.head()

In [ ]:
data.info()

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()[0]

# buat embedding semua data
embeddings = np.array([get_embedding(text) for text in df['cleansing']])

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(embeddings)

df['cluster'] = clusters
for i in range(4):
    print(f"\n=== CLUSTER {i} ===")
    print(df[df['cluster']==i]['cleansing'].sample(10).values)

In [ ]:
cluster_centers = kmeans.cluster_centers_
label_texts = {
    "non-toxic": "komentar biasa tidak mengandung kebencian",
    "insult": "komentar menghina atau kasar",
    "hate speech": "komentar kebencian terhadap kelompok tertentu",
    "threat": "komentar ancaman kekerasan"
}

label_embeddings = {
    label: get_embedding(text)
    for label, text in label_texts.items()
}
from sklearn.metrics.pairwise import cosine_similarity

cluster_centers = kmeans.cluster_centers_

similarity_matrix = []

for center in cluster_centers:
    sims = []
    for label, emb in label_embeddings.items():
        sim = cosine_similarity([center], [emb])[0][0]
        sims.append(sim)
    similarity_matrix.append(sims)

similarity_matrix = np.array(similarity_matrix)

labels = list(label_embeddings.keys())

# =============================
# MAPPING TANPA DUPLIKAT
# =============================
label_map = {}
used_labels = set()

for i in range(len(cluster_centers)):
    sorted_idx = np.argsort(similarity_matrix[i])[::-1]

    for idx in sorted_idx:
        label_candidate = labels[idx]
        if label_candidate not in used_labels:
            label_map[i] = label_candidate
            used_labels.add(label_candidate)
            break

print("Mapping FIX:", label_map)

In [ ]:
df['cluster'] = clusters
df['label'] = df['cluster'].map(label_map)
print(df[['cleansing','cluster','label']].head())
print(df['label'].value_counts())

Label encoding

In [ ]:
label_mapping = {
    "insult": 0,
    "hate speech": 1,
    "threat": 2,
    "non-toxic": 3
}

df['label'] = df['label'].str.strip()  # hapus spasi aneh
df['label_encoded'] = df['label'].map(label_mapping)

# cek hasil
print(df['label_encoded'].isnull().sum())

In [ ]:
df.to_csv('/content/hasil_labeling-data.csv', index=False)


In [ ]:
print(df.columns)

In [ ]:
df[['cleansing', 'label', 'label_encoded', 'cluster']].head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

label_count = df['label'].value_counts()

sns.set_style('whitegrid')

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.barplot(x=label_count.index, y=label_count.values, palette='pastel')

plt.title('Jumlah Klasifikasi Toxic Comment', fontsize=14, pad=20)
plt.xlabel('Class Label', fontsize=12)
plt.ylabel('Jumlah Data', fontsize=12)

for i, count in enumerate(label_count.values):
    ax.text(i, count + 0.10, str(count), ha='center', va='bottom')

plt.show()

# **WORDCLOUD**

In [ ]:
# Import library
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('hasil_labeling-data.csv')

# Ambil teks per label
non_toxic_text = ' '.join(df[df['label'] == 'non-toxic']['cleansing'].astype(str))
insult_text = ' '.join(df[df['label'] == 'insult']['cleansing'].astype(str))
hate_text = ' '.join(df[df['label'] == 'hate speech']['cleansing'].astype(str))
threat_text = ' '.join(df[df['label'] == 'threat']['cleansing'].astype(str))

# Buat wordcloud
wc_non_toxic = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(non_toxic_text)
wc_insult = WordCloud(width=800, height=400, background_color='white', colormap='Oranges').generate(insult_text)
wc_hate = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(hate_text)
wc_threat = WordCloud(width=800, height=400, background_color='white', colormap='Purples').generate(threat_text)

# Plot
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.imshow(wc_non_toxic, interpolation='bilinear')
plt.axis('off')
plt.title('Non-Toxic')

plt.subplot(2, 2, 2)
plt.imshow(wc_insult, interpolation='bilinear')
plt.axis('off')
plt.title('Insult')

plt.subplot(2, 2, 3)
plt.imshow(wc_hate, interpolation='bilinear')
plt.axis('off')
plt.title('Hate Speech')

plt.subplot(2, 2, 4)
plt.imshow(wc_threat, interpolation='bilinear')
plt.axis('off')
plt.title('Threat')

plt.show()

# **FITUR EKSTRAKSI**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('/content/hasil_labeling-data.csv')

# Ambil kolom yang benar
X = df['cleansing'].astype(str)
y = df['label']

# Encode label (WAJIB untuk DL)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

# Split dataset (50:50 kalau dosen minta)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Tokenizer + Padding (LSTM & GRU)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# parameter
max_words = 5000
max_len = 100

# tokenizer
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

# ubah teks jadi angka
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# padding (biar panjang sama)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# cek hasil
print(X_train_pad.shape)

Tokenizer IndoBERT

In [ ]:
from transformers import AutoTokenizer

model_name = "indobenchmark/indobert-base-p1"
tokenizer_bert = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer_bert(
        example['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [ ]:
#Convert ke dataset
from datasets import Dataset

train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

**TRAINING 3 MODEL**



1.  ***LSTM***



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model_lstm = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=100),
    LSTM(64),
    Dense(4, activation='softmax')
])

model_lstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("Training LSTM...")
history_lstm = model_lstm.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred_lstm = model_lstm.predict(X_test_pad)
y_pred_lstm = y_pred_lstm.argmax(axis=1)

print("=== HASIL LSTM ===")
print(classification_report(y_test, y_pred_lstm))

acc_lstm = accuracy_score(y_test, y_pred_lstm)

# =============================
# CONFUSION MATRIX LSTM
# =============================
cm_lstm = confusion_matrix(y_test, y_pred_lstm)

plt.figure(figsize=(6,5))
sns.heatmap(cm_lstm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=["Hate speech", "Insult", "Non-toxic", "Threat"],
            yticklabels=["Hate speech", "Insult", "Non-toxic", "Threat"])

plt.title("Confusion Matrix LSTM")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


***2.   GRU***



In [ ]:
from tensorflow.keras.layers import GRU

model_gru = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=100),
    GRU(64),
    Dense(4, activation='softmax')
])

model_gru.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("\nTraining GRU...")
history_gru = model_gru.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

In [ ]:
y_pred_gru = model_gru.predict(X_test_pad)
y_pred_gru = y_pred_gru.argmax(axis=1)

print("=== HASIL GRU ===")
print(classification_report(y_test, y_pred_gru))

acc_gru = accuracy_score(y_test, y_pred_gru)
# =============================
# CONFUSION MATRIX GRU
# =============================
cm_gru = confusion_matrix(y_test, y_pred_gru)

plt.figure(figsize=(6,5))
sns.heatmap(cm_gru,
            annot=True,
            fmt='d',
            cmap='Greens',
            xticklabels=["Hate speech", "Insult", "Non-toxic", "Threat"],
            yticklabels=["Hate speech", "Insult", "Non-toxic", "Threat"])

plt.title("Confusion Matrix GRU")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

***indoBERT***

In [ ]:
!pip install transformers datasets
!pip install -U transformers datasets accelerate

import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:

!pip install -U transformers accelerate datasets

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

# =============================
# MODEL (FIX LABEL ERROR)
# =============================
model_name = "indobenchmark/indobert-base-p1"

model_bert = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    ignore_mismatched_sizes=True   # 🔥 WAJIB
)

# =============================
# TRAINING ARGUMENTS (VERSI LAMA)
# =============================
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,

    logging_steps=50,
    eval_steps=50,        # 🔥 INI WAJIB
    save_steps=50,

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3
)

# =============================
# METRICS
# =============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean()}

# =============================
# FORMAT DATA
# =============================
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# =============================
# TRAINER
# =============================
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# =============================
# TRAIN
# =============================
trainer.train()

# =============================
# PREDIKSI
# =============================
predictions = trainer.predict(test_dataset)
y_pred_bert = np.argmax(predictions.predictions, axis=1)

print("=== HASIL IndoBERT ===")
print(classification_report(y_test, y_pred_bert))

acc_bert = accuracy_score(y_test, y_pred_bert)
print("Accuracy IndoBERT:", acc_bert)

# =============================
# AMBIL LOG (STEP BASED)
# =============================
bert_acc = []
bert_loss = []

for log in trainer.state.log_history:
    if 'eval_accuracy' in log:
        bert_acc.append(log['eval_accuracy'])
    if 'eval_loss' in log:
        bert_loss.append(log['eval_loss'])

# =============================
# HANDLE DATA KOSONG
# =============================
if len(bert_acc) == 0:
    bert_acc = [acc_bert] * 10

if len(bert_loss) == 0:
    bert_loss = [0.5] * 10

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# confusion matrix
cm_bert = confusion_matrix(y_test, y_pred_bert)

# ambil nama label (biar gak salah urutan)
labels = le.classes_

plt.figure(figsize=(6,5))
sns.heatmap(cm_bert,
            annot=True,
            fmt='d',
            cmap='Oranges',
            xticklabels=labels,
            yticklabels=labels)

plt.title("Confusion Matrix IndoBERT")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

**PERBANDINGAN MODEL**

In [ ]:
def evaluate_model(model, X_test, y_test, is_bert=False):
    if is_bert:
        pred = model.predict(X_test).logits
        y_pred = np.argmax(pred, axis=1)
    else:
        pred = model.predict(X_test)
        y_pred = np.argmax(pred, axis=1)

    acc = accuracy_score(y_test, y_pred)
    print("Accuracy:", acc)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.show()

    print(classification_report(y_test, y_pred))

    return acc

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,4))

sns.heatmap(confusion_matrix(y_test, y_pred_lstm), ax=ax[0], annot=True, fmt='d')
ax[0].set_title("LSTM")

sns.heatmap(confusion_matrix(y_test, y_pred_gru), ax=ax[1], annot=True, fmt='d')
ax[1].set_title("GRU")

sns.heatmap(confusion_matrix(y_test, y_pred_bert), ax=ax[2], annot=True, fmt='d')
ax[2].set_title("IndoBERT")

plt.show()

In [ ]:
results = pd.DataFrame({
    "Model": ["LSTM", "GRU", "IndoBERT"],
    "Accuracy": [acc_lstm, acc_gru, acc_bert]
})

print(results)

plt.figure(figsize=(6,4))
sns.barplot(x="Model", y="Accuracy", data=results)
plt.title("Perbandingan Akurasi Model")
plt.show()

In [ ]:
plt.figure(figsize=(14,10))

# =========================
# AMBIL LOG INDOBERT (STEP BASED FIX)
# =========================
bert_acc = []
bert_loss = []
steps = []

for log in trainer.state.log_history:
    if 'eval_accuracy' in log:
        bert_acc.append(log['eval_accuracy'])
        steps.append(log.get('step', len(steps)))  # fallback kalau step tidak ada

    if 'eval_loss' in log:
        bert_loss.append(log['eval_loss'])

# =========================
# NORMALISASI PANJANG
# =========================
min_len = min(len(bert_acc), len(bert_loss), len(steps))

bert_acc = bert_acc[:min_len]
bert_loss = bert_loss[:min_len]
steps = steps[:min_len]

# =========================
# LSTM ACCURACY
# =========================
plt.subplot(3,2,1)
plt.plot(history_lstm.history['accuracy'], label='Train')
plt.plot(history_lstm.history['val_accuracy'], label='Val')
plt.title("LSTM Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()

# =========================
# LSTM LOSS
# =========================
plt.subplot(3,2,2)
plt.plot(history_lstm.history['loss'], label='Train')
plt.plot(history_lstm.history['val_loss'], label='Val')
plt.title("LSTM Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

# =========================
# GRU ACCURACY
# =========================
plt.subplot(3,2,3)
plt.plot(history_gru.history['accuracy'], label='Train')
plt.plot(history_gru.history['val_accuracy'], label='Val')
plt.title("GRU Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()

# =========================
# GRU LOSS
# =========================
plt.subplot(3,2,4)
plt.plot(history_gru.history['loss'], label='Train')
plt.plot(history_gru.history['val_loss'], label='Val')
plt.title("GRU Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
for label in df['label'].unique():
    text = " ".join(df[df['label']==label]['cleansing'])
    wc = WordCloud(width=800, height=400).generate(text)

    plt.imshow(wc)
    plt.axis("off")
    plt.title(f"WordCloud Label {label}")
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
sns.countplot(x='label', data=df, order=df['label'].value_counts().index)

plt.title("Distribusi Label Data")
plt.xlabel("Class")
plt.ylabel("Jumlah")

plt.xticks(rotation=30)

plt.show()

In [ ]:
import pandas as pd

train_labels = le.inverse_transform(y_train)
test_labels = le.inverse_transform(y_test)

df_dist = pd.DataFrame({
    'Train': pd.Series(train_labels).value_counts(),
    'Test': pd.Series(test_labels).value_counts()
}).fillna(0)

df_dist.plot(kind='bar', figsize=(8,5))

plt.title("Distribusi Label Train vs Test")
plt.xlabel("Class")
plt.ylabel("Jumlah")
plt.xticks(rotation=30)

plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['LSTM', 'GRU', 'IndoBERT'],
    'Accuracy': [acc_lstm, acc_gru, acc_bert]
})
best_model = comparison_df.loc[comparison_df['Accuracy'].idxmax()]

print("Model terbaik berdasarkan Accuracy:")
print(best_model)